### Purpose: Create COBRA inputs for test grids where emissions information and grid information are separate
#### currently file is set up for the New England test grid
#### functions:
    create_samples:
        inputs: number of samples to create, seed for rng
    outputs: 
        DataFile:contains info associatied with the emissions scenario, 
                ie: power level of generator, emissions levels of each pollutant, generator info
        EmissionsScenario: a COBRA input file
    create_batch_file:
        inputs: number of samples to generate
        outputs: a single text file with scripts to run all of the samples.
        Note: must change extension of file to .bat before running



In [1]:
%cd ..
%cd ..

/Users/elizabethrogers/Desktop/powersystemspublichealth/BatchFileGeneration
/Users/elizabethrogers/Desktop/powersystemspublichealth


In [ ]:
import pandas as pd
from pandas import DataFrame, concat
import numpy as np
import random as rand
import os

In [ ]:
# Set up directory locations:
batch_file_dir = 'BatchFileGeneration'
new_england_generation_dir = 'NewEnglandFiles'
cobra_data_dir = 'cobra data dictionary'

Create COBRA input samples using the create_samples function

In [ ]:
def create_samples(num_samples,seed):
    rand.seed(seed)
    print("starting...")
    for j in range(num_samples):
        file = pd.read_csv(os.path.join(batch_file_dir,new_england_generation_dir,"GenericBatchFileWithInfo.csv"))

        generic_file = pd.read_csv(os.path.join(batch_file_dir,new_england_generation_dir,"GenericBatchFileALL.csv"))
        # file containing emissions rates for each 
        em_info = pd.read_csv(os.path.join(batch_file_dir,new_england_generation_dir,"StateRatesInfo.csv"))


        # target_array = target_file.to_numpy()
        
        # store emissions rate constants
        # em_info = em_info.to_numpy()

        # test = target_file

        # save the values necessary for calculating emissions rates
        fuel_type = file['Fuel Type'].to_numpy()
        min_MW = file['Min MW'].to_numpy()
        max_MW = file['Max MW'].to_numpy()
        state = file['stid'].to_numpy()

        # create empty arrays for each rate we want to calculate
        No2 =[]
        So2 = []
        Pm25 = []
        VoC = []
        MWs = []


        # def calc_rates():
        for i in range(len(fuel_type)):
            # sample random generation amount
            MW = rand.uniform(min_MW[i],max_MW[i])

            MWs.append(MW)


            # print(state[i])
            em = em_info.loc[em_info['State ID'] == state[i], ['Fuel Type','Emissions Type', 'Emissions Rate']]
            em = em.loc[em['Fuel Type'] == fuel_type[i], ['Emissions Type', 'Emissions Rate']].to_numpy()
            
            # print(em)

            # divide by 2000 to convert emissions to tons (currently in lbs)
            No2.append(MW * em[0][1] / 2000)
            So2.append(MW * em[1][1] / 2000)
            Pm25.append(MW * em[2][1] / 2000)
            VoC.append(MW * em[3][1] / 2000)


        # insert relevant values
        file.insert(8, 'NOx', No2) # Insert 'C' at index 1 (second column)
        file.insert(9, 'SO2', So2) # Insert 'C' at index 1 (second column)
        file.insert(10,'NH3',[0]*len(No2))
        file.insert(11,'SOA',[0]*len(No2))
        file.insert(12, 'PM25', Pm25) # Insert 'C' at index 1 (second column)
        file.insert(13, 'VOC', VoC) # Insert 'C' at index 1 (second column)
        file.insert(14, 'MW', MWs)


        # # remove extra rows not in scenario file
        final_file = file.drop('Min MW', axis=1) 
        final_file = final_file.drop('Max MW', axis=1) 
        final_file = final_file.drop('MW', axis=1)
        final_file = final_file.drop('Bus number', axis=1) 
        final_file = final_file.drop('Fuel Type', axis=1)

        final_file = final_file.insert(0,'ID', [2]*len(No2))

        # print(final_file.head())
        # print(final_file.head())

        # Save DataFrame to a CSV file
        name1 = 'DataFile' + str(j + 1) + '.csv'
        # name1 = 'testDataFile.csv'
        file.to_csv(name1, index=False)


        # target_array = target_array.astype(str)

        # final_target_file = pd.DataFrame(target_array, columns =['ID','typeindx','sourceindx','stid','cyid','TIER1','TIER2','TIER3','NOx','SO2','NH3','SOA','PM25','VOC'])

        final_target_file = pd.concat([final_file, generic_file])

        # print(final_target_file.head())

        name = 'Emissions_Scenario' + str(j + 1) + '.csv'
        # name = 'testScenario.csv'

        # final_target_file.astype(str)

        final_target_file.to_csv(name,index=False)



In [ ]:
num_samples = 1
seed = 0

In [ ]:
create_samples(num_samples,seed)

## Create Batch File for generated experiments

## Note: after running, change extension of txt batch file to be .bat

In [ ]:
# Set up directories for filepaths of batch file
# I saved my scenario and outcome files locally, hence the filepaths
COBRA_scenario_files_dir = "\"C:\\Users\\Loaner\\Desktop\\powersystemspublichealth\\EmissionsScenarios\\Scenarios082325"
COBRA_experiment_outcomes_dir =  "\"C:\\Users\\Loaner\\Desktop\\COBRA Outcomes\\Experiment 082325"

In [ ]:
for i in range(num_samples):
    filename = "batch"+str(i+1) + ".txt"
    with open(filename, "w") as file:
        # change this
        scenario = COBRA_scenario_files_dir + "\\Emissions_Scenario" + str(i+1)+".csv\""
        outcome = COBRA_experiment_outcomes_dir + "\\Outcome"+str(i+1)+".csv\""
        # change baseline file
        baseline = os.path.join(batch_file_dir,new_england_generation_dir,"NewEngland_BaselineEmissions_2023.csv")
        # "\"C:\\Users\\Loaner\\COBRA\\input files\\default data\\default_2023_population_data.csv\""
        population = os.path.join(batch_file_dir,cobra_data_dir,"2023_default_population_data.csv")
        incidence_data = "\"C:\\Users\\Loaner\\COBRA\\input files\\default data\\default_2023_incidence_data.csv\""
        valuation_data = "\"C:\\Users\\Loaner\\COBRA\\input files\\default data\\default_2023_valuation_data.csv\""
        # filename = "\"C:\\Users\\elrog\\COBRA\\cobra_console.exe\" -d \"C:\\Users\\elrog\\COBRA\\data\\cobra.db\" -b " + baseline + " -c "+scenario+ " -p \"C:\\Users\\elrog\\COBRA\\input files\\new data\\default_2023_population_data.csv\" -i \"C:\\Users\\elrog\\COBRA\\input files\\default_2023_incidence_data.csv\" -v \"C:\\Users\\elrog\\COBRA\\input files\\new data\\default_2023_valuation_data.csv\" -o "+outcome+ " --discountrate 2 \n \n"        

        file.write("\"C:\\Users\\Loaner\\COBRA\\cobra_console.exe\" -d \"C:\\Users\\Loaner\\COBRA\\data\\cobra.db\" -b " + baseline + " -c "+scenario+ " -p " + population + " -i " + incidence_data + " -v " + valuation_data +" -o "+outcome+ " --discountrate 2 \n \n")              
        # file_index+=1
        # print(filename)